# Phase Retrieval Baseline: Error Reduction (ER) and Hybrid Input-Output (HIO)

This notebook demonstrates classical iterative phase retrieval on an OpenDiffraction sample.

### Mathematical Background
In Coherent Diffraction Imaging (CDI), we measure the diffraction intensity $I(\mathbf{q}) = |A(\mathbf{q})|^2$, losing the phase $\arg(A(\mathbf{q}))$. To reconstruct the real-space object $O(\mathbf{r})$, we enforce:
1. **Fourier Constraint**: $|\mathcal{F}[g_k(\mathbf{r})]| = \sqrt{I(\mathbf{q})}$ on valid detector pixels.
2. **Real-Space Support Constraint**: $g(\mathbf{r}) = 0$ outside known finite support $S$, with non-negativity $g(\mathbf{r}) \ge 0$.

### Phase Retrieval Ambiguities
Any reconstruction from intensity modulus alone is subject to fundamental trivial ambiguities:
- **Global phase shift**: $O(\mathbf{r}) \to O(\mathbf{r}) e^{i \phi_0}$
- **Spatial translation**: $O(\mathbf{r}) \to O(\mathbf{r} - \mathbf{r}_0)$
- **Twin-image (conjugate inversion)**: $O(\mathbf{r}) \to O^*(-\mathbf{r})$

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

from opendiffraction.io.loader import OpenDiffractionDataset
from opendiffraction.physics.phase_retrieval import error_reduction, hybrid_input_output

dataset_dir = "../output"
# Select a clean coherent sample (Profile V01)
ds_clean = OpenDiffractionDataset(dataset_dir, variation="V01")
if len(ds_clean) == 0:
    # Fallback to any sample if V01 not filtered
    ds_clean = OpenDiffractionDataset(dataset_dir)

sample = ds_clean[0]
print(f"Selected sample: {sample.sample_id} ({sample.metadata.get('variation_family', 'N/A')})")

## 1. Preparing Measured Modulus and Loose Support

In practical experimental phase retrieval, a loose support is defined (e.g. by shrinkwrap or a bounding box).

In [ ]:
meas_inten = sample.intensity_ideal[0]
measured_modulus = np.sqrt(np.maximum(meas_inten, 0))

# Create support mask in 512x512 detector frame
# Center 64x64 region plus a 6-pixel dilation margin
support = np.zeros((512, 512), dtype=bool)
support[256 - 38 : 256 + 38, 256 - 38 : 256 + 38] = True

gt_obj = sample.object_amplitude[0]

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(gt_obj, cmap="inferno", origin="lower")
axes[0].set_title("Ground Truth Object (64x64)")
axes[1].imshow(support[256-45:256+45, 256-45:256+45], cmap="gray", origin="lower")
axes[1].set_title("Loose Support Mask (Crop)")
plt.show()

## 2. Iterative Reconstruction: HIO Followed by ER

We run 40 iterations of Hybrid Input-Output ($\beta = 0.8$) to escape local minima, followed by 20 iterations of Error Reduction to converge.

In [ ]:
rng = np.random.default_rng(42)

# Run HIO
hio_recon, hio_errors = hybrid_input_output(
    measured_amplitude=measured_modulus,
    support_mask=support,
    num_iterations=40,
    beta=0.8,
    rng=rng,
)

# Refine with ER
er_recon, er_errors = error_reduction(
    measured_amplitude=measured_modulus,
    support_mask=support,
    num_iterations=20,
    initial_phase=None,
    rng=rng,
)

# Crop reconstruction to 64x64 central ROI
crop_recon = er_recon[256 - 32 : 256 + 32, 256 - 32 : 256 + 32]

# Check twin-image orientation
crop_recon_twin = np.flip(crop_recon)
err_normal = np.linalg.norm(crop_recon / (crop_recon.max() + 1e-6) - gt_obj / (gt_obj.max() + 1e-6))
err_twin = np.linalg.norm(crop_recon_twin / (crop_recon_twin.max() + 1e-6) - gt_obj / (gt_obj.max() + 1e-6))

final_recon = crop_recon_twin if err_twin < err_normal else crop_recon
is_twin = err_twin < err_normal
print(f"Reconstruction matched: {'Twin Image (Inverted)' if is_twin else 'Standard Orientation'}")

## 3. Results Comparison

We compare the ground-truth object against the reconstructed amplitude and plot the Fourier error convergence.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(gt_obj, cmap="inferno", origin="lower")
axes[0].set_title("Ground Truth $O(r)$")
axes[0].axis("off")

axes[1].imshow(final_recon, cmap="inferno", origin="lower")
axes[1].set_title("Reconstruction (ER/HIO)")
axes[1].axis("off")

axes[2].plot(hio_errors, label="HIO ($\beta=0.8$)", color="tab:blue")
axes[2].plot(er_errors, label="ER", color="tab:orange")
axes[2].set_yscale("log")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Fourier Error $R^2$")
axes[2].set_title("Convergence Trajectory")
axes[2].legend()

plt.tight_layout()
plt.show()